# Calcular DDT de una S-box

Edita el array `sbox` en la primera celda y ejecuta las celdas para calcular y mostrar la Differential Distribution Table.

In [13]:
# Ejemplo: S-box PRESENT de 4 bits.
sbox = [
    0xC, 0x5, 0x6, 0xB,
    0x9, 0x0, 0xA, 0xD,
    0x3, 0xE, 0xF, 0x8,
    0x4, 0x7, 0x1, 0x2,
]

# Deja None para inferirlos automaticamente.
input_size = None
output_size = None

In [14]:
sbox = [
    0x1, 0x2, 0x0, 0x3,
    0x4, 0x6, 0x5, 0x7
]

# Deja None para inferirlos automaticamente.
input_size = None
output_size = None

In [15]:
sbox = [
    0x3, 0x1, 0x4, 0x5,
    0x2, 0x7, 0x0, 0x6
]

# Deja None para inferirlos automaticamente.
input_size = None
output_size = None

In [16]:
sbox = [
    0x1, 0x2, 0x3, 0x4,
    0x5, 0x6, 0x7, 0x0
]

# Deja None para inferirlos automaticamente.
input_size = None
output_size = None

In [17]:
sbox = [
    0xE, 0x4, 0xD, 0x1,
    0x2, 0xF, 0xB, 0x8,
    0x3, 0xA, 0x6, 0xC,
    0x5, 0x9, 0x0, 0x7
]

In [18]:
import math
import pandas as pd


def infer_input_size(table):
    if len(table) == 0 or len(table) & (len(table) - 1):
        raise ValueError("La longitud de la S-box debe ser una potencia de 2")
    return int(math.log2(len(table)))


def infer_output_size(table):
    return max(1, max(int(v).bit_length() for v in table))


def compute_ddt(table, n_bits_in=None, n_bits_out=None):
    table = [int(v) for v in table]
    n_bits_in = infer_input_size(table) if n_bits_in is None else n_bits_in
    n_bits_out = infer_output_size(table) if n_bits_out is None else n_bits_out

    n_inputs = 1 << n_bits_in
    n_outputs = 1 << n_bits_out

    if len(table) != n_inputs:
        raise ValueError(f"La S-box debe tener {n_inputs} entradas para input_size={n_bits_in}")
    if any(v < 0 or v >= n_outputs for v in table):
        raise ValueError(f"Todos los valores deben estar entre 0 y {n_outputs - 1}")

    ddt = [[0 for _ in range(n_outputs)] for _ in range(n_inputs)]
    for dx in range(n_inputs):
        for x in range(n_inputs):
            dy = table[x] ^ table[x ^ dx]
            ddt[dx][dy] += 1

    return ddt, n_bits_in, n_bits_out


ddt, input_size, output_size = compute_ddt(sbox, input_size, output_size)

row_labels = [f"dx={i:#x}" for i in range(1 << input_size)]
col_labels = [f"dy={i:#x}" for i in range(1 << output_size)]
ddt_df = pd.DataFrame(ddt, index=row_labels, columns=col_labels)

print(f"input_size={input_size}, output_size={output_size}")
ddt_df

input_size=4, output_size=4


,dy=0x0,dy=0x1,dy=0x2,dy=0x3,dy=0x4,dy=0x5,dy=0x6,dy=0x7,dy=0x8,dy=0x9,dy=0xa,dy=0xb,dy=0xc,dy=0xd,dy=0xe,dy=0xf
dx=0x0,16,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
dx=0x1,0,0,0,2,0,0,0,2,0,2,4,0,4,2,0,0
dx=0x2,0,0,0,2,0,6,2,2,0,2,0,0,0,0,2,0
dx=0x3,0,0,2,0,2,0,0,0,0,4,2,0,2,0,0,4
dx=0x4,0,0,0,2,0,0,6,0,0,2,0,4,2,0,0,0
dx=0x5,0,4,0,0,0,2,2,0,0,0,4,0,2,0,0,2
dx=0x6,0,0,0,4,0,4,0,0,0,0,0,0,2,2,2,2
dx=0x7,0,0,2,2,2,0,2,0,0,2,2,0,0,0,0,4
dx=0x8,0,0,0,0,0,0,2,2,0,0,0,4,0,4,2,2
dx=0x9,0,2,0,0,2,0,0,4,2,0,2,2,2,0,0,0


In [19]:
from pathlib import Path
import sys
import numpy as np


def add_repo_root_to_path():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "evaluation").is_dir() and (path / "sboxes").is_dir():
            if str(path) not in sys.path:
                sys.path.insert(0, str(path))
            return path
    raise RuntimeError("No se encontro la raiz del repositorio desde el directorio actual")


ROOT = add_repo_root_to_path()
print(f"Repo root: {ROOT}")

Repo root: C:\Users\juanc\github\Differential-Cryptanalysis-


In [20]:
from sboxes.sbox import SBox
from evaluation.fixed_points import fixed_ponts
from evaluation.linearity import evaluate
from evaluation.avalanche_criterion import avalanche_criterion
from evaluation.markov_chains.basic_markov_chain import (
    DifferentialMarkovChain,
    create_transition_matrix,
)


sbox_obj = SBox(
    input_size=input_size,
    output_size=output_size,
    table=sbox,
    name="S-box personalizada",
)

avg_sac, sac_per_input, std_per_input, max_std = avalanche_criterion(sbox_obj)
fixed_points = int(fixed_ponts(sbox_obj, detail=True)[1])
max_ddt = int(np.max(ddt))
uniformity = int(np.max(ddt[1:])) if len(ddt) > 1 else max_ddt

transition_matrix = create_transition_matrix(
    np.asarray(sbox_obj.table, dtype=np.int64),
    int(sbox_obj.input_size),
    int(sbox_obj.output_size),
    skip_zero=True,
)
chain = DifferentialMarkovChain(transition_matrix)

metrics_df = pd.DataFrame([
    {
        "name": sbox_obj.name,
        "input_size": int(sbox_obj.input_size),
        "output_size": int(sbox_obj.output_size),
        "uniformity": uniformity,
        "max_ddt": max_ddt,
        "fixed_points": fixed_points,
        "non_linearity": float(evaluate(sbox_obj)),
        "avg_sac": float(avg_sac),
        "max_std_sac": float(max_std),
        "sac_per_input": [float(x) for x in np.round(sac_per_input, 6)],
        "std_sac_per_input": [float(x) for x in np.round(std_per_input, 6)],
        "spectral_gap": float(chain.spectral_gap()),
        "tv_distance_r4": float(chain.tv_distance(4)),
        "max_probability_r4": float(chain.max_probability(4)),
        "mixing_time_eps_1e-3": int(chain.mixing_time_iterative(epsilon=1e-3, max_steps=10_000)),
        "mixing_time_spectral_eps_1e-3": float(chain.mixing_time_spectral(eps=1e-3)),
    }
])

metrics_df

,name,input_size,output_size,uniformity,max_ddt,fixed_points,non_linearity,avg_sac,max_std_sac,sac_per_input,std_sac_per_input,spectral_gap,tv_distance_r4,max_probability_r4,mixing_time_eps_1e-3,mixing_time_spectral_eps_1e-3
0,S-box personalizada,4,4,8,16,0,2.0,0.609375,0.207289,"[0.5625, 0.5625, 0.5625, 0.75]","[0.108253, 0.207289, 0.108253, 0.0]",0.603789,0.045785,0.082031,9,11.0


In [21]:
# Opcional: guardar la DDT en CSV.
# ddt_df.to_csv("mi_ddt.csv")
# metrics_df.to_csv("mi_sbox_metrics_summary.csv", index=False)